In [362]:
import gurobipy as gp
from gurobipy import GRB
from gurobipy import *

In [363]:
# parameters
# Operations = [1,2,3,4,5,6,7]
Operations = [1, 2, 3, 4]
J= len(Operations)
Ressources = [1,2,3,4,5,6,7,8]
R= len(Ressources)
T= 16

predecessors = [[],             # Step 1: Station 0_Input
                [0],            # Step 2: Station 1_stearing
                [0],            # Step 2: Station 2_stearing
                [0],            # Step 2: Station 3_stearing
                [1, 2, 3],      # Step 3: Station 4_light
                [1, 2, 3],      # Step 3: Station 5_light
                [1, 2, 3],      # Step 3: Station 6_light
                [4, 5, 6]]      # Step 4: Station 7_Output

#duration = [2, 3, 2, 4, 1, 2, 1]
duration = [2, 3, 2, 1]

#demand_capacity =  [[1, 1, 2, 1, 1, 1, 4],
#                    [2, 1, 1, 2, 1, 1, 3],
#                    [1, 2, 1, 1, 1, 1, 1],
#                    [1, 2, 1, 1, 1, 1, 1],
#                    [1, 2, 3, 1, 3, 1, 2],
#                    [1, 2, 1, 1, 1, 1, 2],
#                    [3, 3, 3, 3, 3, 3, 3]]
demand_capacity= [[1, 2, 2, 1, 1, 2, 1], [1, 1, 2, 1, 1, 1, 4], [2, 1, 1, 2, 1, 1, 3], [1, 2, 1, 1, 1, 1, 1]]

FEZ= [0,0,0,0,0,0,0]
SEZ= [15,15,15,15,15,15,15]

capacity= [10,10,10,10,10,10,10]

In [ ]:
# allowed machines
variants= {0: [[0], [1,2,3], [5], [7]], 1: [[0], [1,2,3], [4,6], [7]], 2: [[0], [1], [6], [7]]}
allowed_machines= {}

for v in variants:
    for j in range(len(variants[v])):
        allowed_machines.update({(v,j): variants[v][j]})

In [366]:
m = gp.Model("RCPSP")

In [367]:
# decision variables
S = m.addVars(J, T, vtype= GRB.BINARY)
Y = m.addVars(J, R, vtype=GRB.BINARY)
C = m.addVar(lb= 0, vtype= GRB.CONTINUOUS)

In [368]:
m.setObjective(C, GRB.MINIMIZE)

In [369]:
# time constraint
for j in range(J):
    m.addConstr(C >= sum((t + duration[j]) * S[j,t] for t in range(FEZ[j], SEZ[j]+1)))

In [370]:
# timeslot constraint
# the job needs to be finished in a certain timeframe
for j in range(J):
    m.addConstr((quicksum(S[j, t] for t in range(FEZ[j], SEZ[j]+1)) == 1))

In [371]:
# variant_based useable machines
# variants are allowed to be produced on a predefined set of machines
for j in range(J):
    m.addConstr(quicksum(Y[j,r] for r in allowed_machines[(v,j)]) == 1)

In [372]:
# variant_based
# forbid the execution of jobs on machines which aren't in the set
for j in range(J):
    for r in range(R):
        if r not in allowed_machines[(v,j)]:
            m.addConstr(Y[j,r] == 0)


In [373]:
# precendence constraint
# realise the order of the correct process
for j in range(J):
        for h in predecessors[j]:
                m.addConstr(quicksum(t * S[h, t] for t in range(FEZ[h], SEZ[h]+1)) <= quicksum((t - duration[h]) * S[j, t] for t in range(FEZ[j], SEZ[j]+1)))

In [374]:
# capacity constraint
# the defined capacity needs to be kept
for r in range(R):
    for t in range(T):
        m.addConstr((quicksum(Y[j,r] * quicksum(S[j,q] for q in range(t, min(t+duration[j], T))) for j in range(J)) <= 1))

In [375]:
# Solve
m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: Intel(R) Core(TM) i7-9750H CPU @ 2.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 43 rows, 97 columns and 254 nonzeros
Model fingerprint: 0xcf1bf033
Model has 128 quadratic constraints
Variable types: 1 continuous, 96 integer (96 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  QMatrix range    [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
  QRHS range       [1e+00, 1e+00]
Presolve removed 43 rows and 97 columns
Presolve time: 0.01s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.00 work units)
Thread count was 1 (of 12 available processors)

Solution count 1: 5 

Optimal solution found (tolerance 1.00e-04)
Best objective 5.000000000000e+00, best bound 5.000000000

In [376]:
#m.computeIIS()
#m.write("rcpsp.ilp")

In [377]:
print("Objective value: ", m.objVal , " found after ", m.Runtime, " seconds. Relative gap is: ", m.MIPGap)

Objective value:  5.0  found after  0.023000001907348633  seconds. Relative gap is:  0.0
